In [0]:
import requests
import json
import os
import time
import uuid

from datetime import date, datetime, timezone
from pyspark.sql import functions as F

In [0]:
# Full load는 recurring incremental ingestion과 분리한다.
# Raw API 응답은 run_id 단위로 보존하며,
# checkpoint를 이용해 수집 중단 시 마지막 성공 page부터 재개할 수 있게 한다.

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

PAGE_SIZE = 1000
MAX_RETRIES = 5

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

full_load_started_at = datetime.now(timezone.utc)
full_load_started_date = full_load_started_at.date().isoformat()

run_id = str(uuid.uuid4())

print("Run ID:", run_id)
print("Full load started at:", full_load_started_at)

Run ID: 37961472-4d89-4e4f-a0fc-2d623119806e
Full load started at: 2026-09-26 09:21:41.598711+00:00


In [0]:
full_raw_dir = (
    f"/Volumes/{catalog}/clinical_trials/raw_api/"
    f"full_load/"
    f"run_date={full_load_started_date}/"
    f"run_id={run_id}"
)

os.makedirs(full_raw_dir, exist_ok=True)

checkpoint_path = f"{full_raw_dir}/checkpoint.json"

print(full_raw_dir)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5503926893171303>, line 2
      1 full_raw_dir = (
----> 2     f"/Volumes/{catalog}/clinical_trials/raw_api/"
      3     f"full_load/"
      4     f"run_date={full_load_started_date}/"
      5     f"run_id={run_id}"
      6 )
      8 os.makedirs(full_raw_dir, exist_ok=True)
     10 checkpoint_path = f"{full_raw_dir}/checkpoint.json"

NameError: name 'catalog' is not defined

In [0]:
def save_checkpoint(
    pages_completed,
    total_studies,
    next_page_token,
    completed=False,
    expected_total=None
):
    checkpoint = {
        "run_id": run_id,
        "full_load_started_at": full_load_started_at.isoformat(),
        "pages_completed": pages_completed,
        "total_studies": total_studies,
        "next_page_token": next_page_token,
        "completed": completed,
        "expected_total": expected_total,
        "updated_at": datetime.now(timezone.utc).isoformat()
    }

    with open(
        checkpoint_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            checkpoint,
            f,
            ensure_ascii=False,
            indent=2
        )

def load_checkpoint():
    if not os.path.exists(checkpoint_path):
        return None

    with open(
        checkpoint_path,
        "r",
        encoding="utf-8"
    ) as f:
        return json.load(f)
    
# 약 600번 API를 호출하므로 일시적인 실패에 대비하여 retry 설
def fetch_page(params):
    for attempt in range(1, MAX_RETRIES + 1):

        try:
            response = requests.get(
                BASE_URL,
                params=params,
                timeout=60
            )

            response.raise_for_status()

            return response.json()

        except requests.RequestException as e:

            if attempt == MAX_RETRIES:
                raise

            wait_seconds = 2 ** attempt

            print(
                f"Request failed "
                f"(attempt {attempt}/{MAX_RETRIES}). "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

In [0]:
# 1페이지 테스트
test_data = fetch_page({
    "pageSize": PAGE_SIZE,
    "countTotal": "true",
    "format": "json"
})

print("Studies in first page:", len(test_data.get("studies", [])))
print("Total count:", test_data.get("totalCount"))
print(
    "Has next token:",
    bool(test_data.get("nextPageToken"))
)

test_file = f"{full_raw_dir}/size_test.json"

with open(
    test_file,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        test_data,
        f,
        ensure_ascii=False
    )

size_mb = (
    os.path.getsize(test_file)
    / 1024
    / 1024
)

expected_total = test_data.get("totalCount")

if expected_total:
    estimated_pages = (
        expected_total + PAGE_SIZE - 1
    ) // PAGE_SIZE

    estimated_gb = (
        size_mb * estimated_pages / 1024
    )

    print("First page size MB:", round(size_mb, 2))
    print("Estimated pages:", estimated_pages)
    print(
        "Rough raw size GB:",
        round(estimated_gb, 2)
    )

Studies in first page: 1000
Total count: 604566
Has next token: True
First page size MB: 17.88
Estimated pages: 605
Rough raw size GB: 10.57


In [0]:
import json
import gzip

raw_bytes = json.dumps(
    test_data,
    ensure_ascii=False
).encode("utf-8")

compressed_bytes = gzip.compress(
    raw_bytes,
    compresslevel=6
)

raw_mb = len(raw_bytes) / 1024 / 1024
compressed_mb = len(compressed_bytes) / 1024 / 1024

compression_ratio = compressed_mb / raw_mb

estimated_compressed_gb = (
    compressed_mb * estimated_pages / 1024
)

print("Raw page MB:", round(raw_mb, 2))
print("Compressed page MB:", round(compressed_mb, 2))
print(
    "Compression ratio:",
    round(compression_ratio * 100, 1),
    "%"
)
print(
    "Estimated compressed full size:",
    round(estimated_compressed_gb, 2),
    "GB"
)

os.remove(test_file)

Raw page MB: 17.88
Compressed page MB: 3.42
Compression ratio: 19.1 %
Estimated compressed full size: 2.02 GB


In [0]:
# Full Pagination 코드

checkpoint = load_checkpoint()

if checkpoint:

    if checkpoint["completed"]:
        print("This full load has already completed.")

    pages_completed = checkpoint["pages_completed"]
    total_studies = checkpoint["total_studies"]
    next_page_token = checkpoint["next_page_token"]
    expected_total = checkpoint.get("expected_total")

    page_num = pages_completed + 1

    print(
        f"Resuming from page {page_num}, "
        f"studies already collected: {total_studies}"
    )

else:

    page_num = 1
    total_studies = 0
    next_page_token = None
    expected_total = None

In [0]:
import gzip

while True:
    params = {"pageSize": PAGE_SIZE, "format": "json"}

    # 첫 요청에서 전체 규모 확인
    if page_num == 1:
        params["countTotal"] = "true"

    if next_page_token:
        params["pageToken"] = next_page_token

    # API 호출
    data = fetch_page(params)

    studies = data.get("studies", [])

    if not studies and data.get("nextPageToken"):
        raise RuntimeError(
            f"Page {page_num} returned 0 studies "
            "but still has a nextPageToken."
        )

    # 최초 요청에서 전체 예상 건수 기록
    if expected_total is None:
        expected_total = data.get("totalCount")

    # 현재 페이지 Raw JSON을 gzip으로 저장
    file_path = f"{full_raw_dir}/page_{page_num:06d}.json.gz"

    with gzip.open(file_path, "wt", encoding="utf-8", compresslevel=6) as f:
        json.dump(data, f, ensure_ascii=False)

    # 여기부터는 gzip 파일 저장이 정상적으로 끝난 뒤 실행
    total_studies += len(studies)

    next_page_token = data.get("nextPageToken")

    # 매 페이지마다 checkpoint 저장
    save_checkpoint(
        pages_completed=page_num,
        total_studies=total_studies,
        next_page_token=next_page_token,
        completed=not bool(next_page_token),
        expected_total=expected_total,
    )

    # 로그는 10페이지마다 출력
    if page_num % 10 == 0 or not next_page_token:
        progress_pct = total_studies / expected_total * 100 if expected_total else 0

        print(
            f"page={page_num:,} | "
            f"total={total_studies:,} | "
            f"progress={progress_pct:.1f}%"
        )

    # 마지막 페이지
    if not next_page_token:
        print("Full load ingestion completed.")
        break

    page_num += 1

    # API에 과도한 연속 요청을 보내지 않도록 짧은 간격
    time.sleep(0.2)

page=10 | total=10,000 | progress=1.7%
page=20 | total=20,000 | progress=3.3%
page=30 | total=30,000 | progress=5.0%
page=40 | total=40,000 | progress=6.6%
page=50 | total=50,000 | progress=8.3%
page=60 | total=60,000 | progress=9.9%
page=70 | total=70,000 | progress=11.6%
page=80 | total=80,000 | progress=13.2%
page=90 | total=90,000 | progress=14.9%
page=100 | total=100,000 | progress=16.5%
page=110 | total=110,000 | progress=18.2%
page=120 | total=120,000 | progress=19.8%
page=130 | total=130,000 | progress=21.5%
page=140 | total=140,000 | progress=23.2%
page=150 | total=150,000 | progress=24.8%
page=160 | total=160,000 | progress=26.5%
page=170 | total=170,000 | progress=28.1%
page=180 | total=180,000 | progress=29.8%
page=190 | total=190,000 | progress=31.4%
page=200 | total=200,000 | progress=33.1%
page=210 | total=210,000 | progress=34.7%
page=220 | total=220,000 | progress=36.4%
page=230 | total=230,000 | progress=38.0%
page=240 | total=240,000 | progress=39.7%
page=250 | total

In [0]:
raw_files = [
    f for f in os.listdir("/Volumes/workspace/clinical_trials/raw_api/full_load/run_date=2026-09-26/run_id=37961472-4d89-4e4f-a0fc-2d623119806e")
    if f.endswith(".json.gz")
]

print("Raw page files:", len(raw_files))

full_pages_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{full_raw_dir}/page_*.json.gz")
)

print("Raw API pages:", full_pages_df.count())

Raw page files: 605
Raw API pages: 605


In [0]:
full_studies_df = (
    full_pages_df
    .select(
        F.explode("studies").alias("study")
    )
)

full_study_count = full_studies_df.count()

print("Full studies:", full_study_count)

Full studies: 604566


In [0]:
full_bronze_staging_df = (
    full_studies_df
    .withColumn(
        "source",
        F.lit("clinicaltrials.gov")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "ingest_date",
        F.lit(full_load_started_date).cast("date")
    )
)

In [0]:
(
    full_bronze_staging_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "clinical_trials.bronze_studies_full_staging"
    )
)

In [0]:
staging_df = spark.read.table(
    "clinical_trials.bronze_studies_full_staging"
)

print(staging_df.count())

null_nct = (
    staging_df
    .filter(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).isNull()
    )
    .count()
)

print("Null NCT IDs:", null_nct)

duplicate_nct_df = (
    staging_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id")
    )
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate NCT IDs:",
    duplicate_nct_df.count()
)

604566
Null NCT IDs: 0
Duplicate NCT IDs: 0


In [0]:
(
    staging_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "clinical_trials.bronze_studies"
    )
)